### Drone Audio With/Without Payload

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as ss
import scipy.fft
import pandas as pd
from scipy.signal.windows import hann
from scipy import signal
import ssqueezepy as sq
from scipy.signal import ShortTimeFFT
from ssqueezepy.experimental import scale_to_freq
import subprocess
import os
from scipy.signal import welch
from matplotlib.backends.backend_pdf import PdfPages

##### Load Data:

In [ ]:
# sampling rate is 48kHz
# Without payload:
with_p, sr1 = librosa.load('With MagArrow.wav', sr=48000)
# With payload:
without_p, sr2 = librosa.load('Without MagArrow.wav', sr=48000)

##### Raw Data Plots:

In [ ]:
# Set up dataframes for raw data:
raw_with = pd.DataFrame(with_p)
raw_with.columns = ['Raw Amplitude']
time_sec1 = raw_with.index/sr1
raw_with['Time (s)'] = time_sec1

raw_without = pd.DataFrame(without_p)
raw_without.columns = ['Raw Amplitude']
time_sec2 = raw_without.index/sr2
raw_without['Time (s)'] = time_sec2
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Plot + save:
with PdfPages('raw_data_with_payload.pdf') as pdf:
    plt.figure(figsize=(12,4))
    plt.plot(raw_with['Time (s)'], raw_with['Raw Amplitude'])
    plt.grid(True, alpha=0.5)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Raw Data - With Payload")
    pdf.savefig()
    plt.show()
    plt.close()
# some stats:
print(ss.describe(raw_with['Raw Amplitude']))
print("standard deviaiton:", np.std(raw_with['Raw Amplitude']))
print("median:", np.median(raw_with['Raw Amplitude']))
print("mode:", ss.mode(raw_with['Raw Amplitude']))
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('raw_data_without_payload.pdf') as pdf:
    plt.figure(figsize=(12,4))
    plt.plot(raw_without['Time (s)'], raw_without['Raw Amplitude'])
    plt.grid(True, alpha=0.5)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Raw Data - Without Payload")
    pdf.savefig()
    plt.show()
    plt.close()
# some stats:
print(ss.describe(raw_without['Raw Amplitude']))
print("standard deviaiton:", np.std(raw_without['Raw Amplitude']))
print("median:", np.median(raw_without['Raw Amplitude']))
print("mode:", ss.mode(raw_without['Raw Amplitude']))

##### Plot Normalized, Detrended, Mean-Shifted (DNS) Data:
- The data is linearly detrended, shifted to approx. zero mean, then normalized
- The below cell also organizes the data into pandas DataFrames
- Plots are created including some summary statistics

In [ ]:
# Detrend with scipy:
with_p_detrended = signal.detrend(with_p, type='linear')
without_p_detrended = signal.detrend(without_p, type='linear')

# Center the data (zero mean)
with_p_centered = with_p_detrended - np.mean(with_p_detrended)
without_p_centered = without_p_detrended - np.mean(without_p_detrended)

# Normalize:
with_p_normalized = librosa.util.normalize(with_p_centered)
without_p_normalized = librosa.util.normalize(without_p_centered)

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Use pandas DataFrames to store DNS data:
with_payload = pd.DataFrame(with_p_normalized)
with_payload.columns = ['Amplitude'] # amplitude column contains DNS data
time_seconds1 = with_payload.index/sr1
with_payload['Time (s)'] = time_seconds1
# other columns for option to plot non-normalized data:
with_payload['detrended_centered'] = with_p_centered
#print(with_payload.head())

without_payload = pd.DataFrame(without_p_normalized)
without_payload.columns = ['Amplitude']
time_seconds2 = without_payload.index/sr2
without_payload['Time (s)'] = time_seconds2
# other columns for option to plot non-normalized data:
without_payload['detrended_centered'] = without_p_centered
#print(without_payload.head())
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('drone_audio_with_payload.pdf') as pdf:
    # Plot both time series:
    plt.figure(figsize=(12,4))
    #plt.plot(with_payload['Time (s)']/60, with_payload['Amplitude'], label='Normalized Drone Audio With Payload') # has time in minutes
    plt.plot(with_payload['Time (s)'], with_payload['Amplitude'], label='Normalized Drone Audio With Payload') # has time in seconds
    plt.title('Drone Audio With Payload')
    plt.xlabel('Time (sec)')
    plt.ylabel('Amplitude')
    plt.grid(True, alpha=0.5)
    #plt.xlim(0, 10)
    plt.tight_layout()
    pdf.savefig()
    plt.show()
    plt.close()

print(f'sampling rate: {sr1} samples per second')
# some stats:
print(ss.describe(with_p_normalized))
print("standard deviaiton:", np.std(with_p_normalized))
print("median:", np.median(with_p_normalized))
print("mode:", ss.mode(with_p_normalized))
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('drone_audio_without_payload.pdf') as pdf:
    plt.figure(figsize=(12,4))
    #plt.plot(without_payload['Time (s)']/60, without_payload['Amplitude'], label='Normalized Drone Audio Without Payload') # has time in minutes
    plt.plot(without_payload['Time (s)'], without_payload['Amplitude'], label='Normalized Drone Audio Without Payload') # has time in seconds
    plt.title('Drone Audio Without Payload')
    plt.xlabel('Time (sec)')
    plt.ylabel('Amplitude')
    plt.grid(True, alpha=0.5)
    #plt.xlim(0, 10)
    plt.tight_layout()
    pdf.savefig()
    plt.show()
    plt.close()

print(f'sampling rate: {sr2} samples per second')
# some stats:
print(ss.describe(without_p_normalized))
print("standard deviaiton:", np.std(without_p_normalized))
print("median:", np.median(without_p_normalized))
print("mode:", ss.mode(without_p_normalized))

In [ ]:
# Make joined result with one time series after the other
# Get the time offset: last time value of raw_with + one sample interval so no overlap
time_offset = raw_with['Time (s)'].iloc[-1] + (1 / sr1)

# Shift the second dataframe's time so it starts right after the first ends
raw_without_shifted = raw_without.copy()
raw_without_shifted['Time (s)'] = raw_without['Time (s)'] + time_offset

# Concatenate the two dataframes (link together)
raw_joined = pd.concat([raw_with, raw_without_shifted], ignore_index=True) # raw_joined is new dataframe

print(raw_joined.shape)
#print(raw_joined.head())
#print(raw_joined.tail())

with PdfPages('joined_data_plot.pdf') as pdf:
    # Plot the joined result
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(raw_joined['Time (s)'] / 60, raw_joined['Raw Amplitude'], linewidth=0.5)
    # Add a vertical line showing where the join happens
    join_time_min = time_offset / 60
    ax.axvline(x=join_time_min, color='red', linestyle='--', linewidth=1.2, label=f'Join point ({join_time_min:.2f} min)')
    ax.set_title('Joined Raw Data - With Payload to Without Payload')
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Amplitude')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    pdf.savefig()
    plt.show()
    plt.close()

##### Align & Trim Data:
- Have the drone sounds begin at the same time based on a threshold amplitude value
    - The point at which the threshold amplitude is exceeded can be limited to a specific time window
- Trim the time series so they end at the same time/are the same length

In [ ]:
def find_start(signal, signal_col, time_col, threshold, search_start=0, search_end=10):
    """signal: pandas dataframe with the signal to analyze
    signal_col: column of interest from signal, str
    time_col: time column corresponding to signal_col
    threshold: absolute amplitude threshold
    search_start: start of time window to search (seconds)
    search_end: end of time window to search (seconds)"""
    # Limit search to time window
    mask = (signal[time_col] >= search_start) & (signal[time_col] <= search_end)
    search_signal = signal[mask].reset_index(drop=True)

    # Find first sample where absolute value exceeds threshold (within window)
    start_idx_local = np.argmax(np.abs(search_signal[signal_col]) > threshold)

    # Verify we actually found a crossing
    if np.abs(search_signal[signal_col][start_idx_local]) <= threshold:
        print(f"Warning: No sample found above threshold {threshold}")
        return 0

    # Map back to index in the original dataframe
    start_idx = signal[mask].index[start_idx_local]

    print("start_idx:", start_idx)

    with PdfPages('find_start_plots.pdf') as pdf:
        # Plot
        plt.figure(figsize=(12, 4))
        plt.plot(signal[time_col], signal[signal_col], label='Time Series')
        plt.axvline(x=signal[time_col].iloc[start_idx], color='red', linestyle='--', label='Detected Start')
        plt.xlim(0, 8)
        plt.grid(True, alpha=0.5)
        plt.legend()
        pdf.savefig()
        plt.show()
        plt.close()

    return start_idx

# Only search between t=5s and t=10s, where you expect the drone to start
start_with = find_start(with_payload, 'Amplitude', 'Time (s)', threshold=0.1, search_start=6, search_end=8)
start_without = find_start(without_payload, 'Amplitude', 'Time (s)', threshold=0.1, search_start=2, search_end=4)

In [ ]:
# Trim and match length:
offset = int(sr1)  # 1 second of samples (assumes sr1 == sr2), use to trim 1 second back from start_idx

start_with_adj = max(0, start_with - offset)      # max(0,...) prevents negative indexing
start_without_adj = max(0, start_without - offset)

min_len = min(len(with_payload) - start_with_adj, len(without_payload) - start_without_adj)

with_payload = with_payload.iloc[start_with_adj:start_with_adj + min_len].reset_index(drop=True)
without_payload = without_payload.iloc[start_without_adj:start_without_adj + min_len].reset_index(drop=True)

# Reset time to start from 0
with_payload['Time (s)'] = with_payload.index / sr1
without_payload['Time (s)'] = without_payload.index / sr2

***
##### Plot Tapered Data:
- A Hann window is used for tapering
- The taper is set to begin at 10% from the edges; this can be adjusted

In [ ]:
def apply_hann_window(data, column, taper_fraction= 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float)
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data.iloc[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data.iloc[-taper_length:] *= right_window
    
    return windowed_data
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with_p_tapered = apply_hann_window(with_payload, 'Amplitude')
with PdfPages('tapered_with_payload.pdf') as pdf:
    plt.figure(figsize=(12,4))
    plt.plot(with_payload['Time (s)']/60, with_payload['Amplitude'], label='Full Data')
    plt.plot(with_payload['Time (s)']/60, with_p_tapered, label='Tapered Data')
    plt.legend(loc='upper right')
    plt.xlabel("Time (min)")
    plt.ylabel("Amplitude")
    plt.title("Full/Tapered Drone Audio With Payload")
    plt.grid(True, alpha=0.5)
    pdf.savefig()
    plt.show()
    plt.close()

without_p_tapered = apply_hann_window(without_payload, 'Amplitude')
with PdfPages('tapered_without_payload.pdf') as pdf:
    plt.figure(figsize=(12,4))
    plt.plot(without_payload['Time (s)']/60, without_payload['Amplitude'], label='Full Data')
    plt.plot(without_payload['Time (s)']/60, without_p_tapered, label='Tapered Data')
    plt.legend(loc='upper right')
    plt.xlabel("Time (min)")
    plt.ylabel("Amplitude")
    plt.title("Full/Tapered Drone Audio Without Payload")
    plt.grid(True, alpha=0.5)
    pdf.savefig()
    plt.show()
    plt.close()

***
#### Fast Fourier Transform:


##### Decrease FFT Frequency Resolution:
- Method: frequency bin averaging

In [ ]:
# Version of plot_averaged_fft with squared magnitudes...
def plot_averaged_fft2(data, column, sr, segment_size, title):
    """Plot FFT with frequency bin averaging 
    data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    segment_size:segment size for averaging, int
    title: plot title, string"""
    with PdfPages(f'{title}.pdf') as pdf:
        # Goal: decrease frequency resolution of the FFT
        # Signal parameters
        T = 1 / sr    # Sampling period
        # signal x:
        x = data[column].values
        N = len(x)
        
        # Parameters for averaging
        overlap = 0.5    # 50% overlap
        step_size = int(segment_size * (1 - overlap))
        
        # Calculate number of segments
        num_segments = (N - segment_size) // step_size + 1
        
        # Initialize array to store summed FFT results
        avg_fft_magnitude = np.zeros(segment_size // 2 + 1) # only need one side for real input
        
        for i in range(num_segments):
            start = i * step_size
            end = start + segment_size
            segment = x[start:end]
            
            # apply a Hanning window to the segment to reduce spectral leakage
            window = np.hanning(segment_size)
            windowed_segment = segment * window
            
            # Compute FFT of the segment
            segment_fft = scipy.fft.rfft(windowed_segment, segment_size)
            # note: here windowed_segment is already same length as segment_size
            #print(len(windowed_segment), segment_size)
            
            # Get the magnitude of the positive frequency side
            segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1] / np.sum(window) # include window sum for amplitude correction
            
            # Accumulate the magnitudes
            #avg_fft_magnitude += segment_magnitude
            avg_fft_magnitude += segment_magnitude ** 2   # accumulate power
        
        # Calculate the mean magnitude
        #avg_fft_magnitude /= num_segments
        avg_fft_magnitude = np.sqrt(avg_fft_magnitude / num_segments) # don't take square root if want power spectral density instead
        
        # Get the frequency bins
        freqs = scipy.fft.rfftfreq(segment_size, T)[:segment_size // 2 + 1]
        
        # Plot the averaged frequency spectrum
        plt.figure(figsize=(12, 6))
        #plt.plot(freqs, avg_fft_magnitude)
        plt.loglog(freqs, avg_fft_magnitude)
        plt.xlabel('Frequency (Hz)', fontsize=12)
        plt.ylabel('Averaged Magnitude', fontsize=12)
        plt.ylim(bottom=10**-4)
        plt.xlim(right=10**4)
        plt.title(title)
        plt.grid(True, which='both', alpha=0.5)
        pdf.savefig()
        plt.show()
        plt.close()
    
        return freqs, avg_fft_magnitude

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Results for DNS data:
print("DNS data result:")
freqs_1, avg_1 = plot_averaged_fft2(with_payload, 'Amplitude', sr1, 8096, title='With Payload - Averaged Frequency Spectrum')
freqs_2, avg_2 = plot_averaged_fft2(without_payload, 'Amplitude', sr2, 8096, title='Without Payload - Averaged Frequency Spectrum')
# Results for Non-Normalized Data:
print("Non-normalized data result:")
freqs__1, avg__1 = plot_averaged_fft2(with_payload, 'detrended_centered', sr1, 4096, title='Drone With Payload - Averaged Frequency Spectrum')
freqs__2, avg__2 = plot_averaged_fft2(without_payload, 'detrended_centered', sr2, 4096, title='Drone Without Payload - Averaged Frequency Spectrum')

In [ ]:
# Plot non-normalized results together:
with PdfPages('avg_fft_with_and_without_payload.pdf') as pdf:
    figu, ax_ = plt.subplots(figsize=(12,6))
    ax_.loglog(freqs__1, avg__1, label='Drone With Payload')
    ax_.loglog(freqs__2, avg__2, label='Drone Without Payload')
    ax_.legend(loc='upper left')
    ax_.grid(True, which='both', alpha=0.5)
    ax_.set_xlabel("Frequency (Hz)")
    ax_.set_ylabel("Averaged Magnitude")
    ax_.set_title("Averaged Frequency Spectrum for Drone With/Without Payload")
    ax_.set_ylim(bottom=10**-4)
    ax_.set_xlim(10, 10**4)
    pdf.savefig()
    plt.show()
    plt.close()

##### Power Spectral Density With Welch's Method:
- Note: The PSD has units of $V^2/Hz$ if e.g. data is measured in V and sr is measured in Hz
    - Unknown units for amplitude from sound recordings, so y axis units left empty
- Change the scaling parameter in welch() from 'density' to 'spectrum' to plot a squared magnitude spectrum instead
    - For a squared magnitude spectrum, the y-axis units would be $V^2$ assuming data is measured in V, for example

In [ ]:
# Trying power spectral density:
def welch_psd(data, column, sr, title, segment_size, window='hann'):
    """data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    title: title for PSD plot, string
    segment_size: length of each segment
    window: desired window to use"""
    with PdfPages(f'{title}.pdf') as pdf:
        overlap = segment_size//2 # number of points to overlap between segments (here is 50%)
        freqs, psd = welch(data[column].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')
    
        # Plot:
        plt.figure(figsize=(12,6))
        plt.loglog(freqs, psd)
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("PSD")
        plt.title(title)
        plt.grid(True, which='both', alpha=0.5)
        plt.ylim(bottom=10**-10)
        pdf.savefig()
        plt.show()
        plt.close()
    
        return freqs, psd

freqs_with, psd_with = welch_psd(with_payload, 'detrended_centered', sr1, 'Drone With Payload PSD', segment_size=4096)
freqs_without, psd_without = welch_psd(without_payload, 'detrended_centered', sr2, 'Drone Without Payload PSD', segment_size=4096)

In [ ]:
# Plot both PSD results together:
with PdfPages('PSD_with_and_without_payload.pdf') as pdf:
    fig, ax = plt.subplots(figsize=(12,6))
    ax.loglog(freqs_with, psd_with, label='With Payload')
    ax.loglog(freqs_without, psd_without, label='Without Payload')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')
    ax.set_ylim(bottom=10**-8)
    ax.set_xlim(10, 10**4)
    ax.set_title('Power Spectral Density For Drone With/Without Payload')
    plt.grid(True, which='both', alpha=0.5)
    ax.legend(loc='upper right')
    pdf.savefig()
    plt.show()
    plt.close()

In [ ]:
# PSD integral to get power:
# print("For Drone With Payload:")
# total_power_with_payload = np.trapz(psd_with, freqs_with)
# print(f"Total power: {total_power_with_payload:.6f}")
# print(f"RMS amplitude: {np.sqrt(total_power_with_payload):.6f}")

# # for specific frequency range:
# f_low, f_high = 10, 500  # Hz, adjust to your region of interest
# mask = (freqs_with >= f_low) & (freqs_with <= f_high)
# band_power = np.trapz(psd_with[mask], freqs_with[mask])
# band_rms = np.sqrt(band_power)
# print(f"Frequency band ({f_low}, {f_high}) power: {band_power:.6f}")
# print(f"Frequency band ({f_low}, {f_high}) RSM amplitude: {band_rms: .6f}")
# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# print("")
# # PSD integral to get power:
# print("For Drone Without Payload:")
# total_power_without_payload = np.trapz(psd_without, freqs_without)
# print(f"Total power: {total_power_without_payload:.6f}")
# print(f"RMS amplitude: {np.sqrt(total_power_without_payload):.6f}")

# # for specific frequency range:
# f_low_, f_high_ = 10, 500  # Hz, adjust to your region of interest
# mask_ = (freqs_without >= f_low_) & (freqs_without <= f_high_)
# band_power_ = np.trapz(psd_without[mask_], freqs_without[mask_])
# band_rms_ = np.sqrt(band_power_)
# print(f"Frequency band ({f_low_}, {f_high_}) power: {band_power_:.6f}")
# print(f"Frequency band ({f_low_}, {f_high_}) RSM amplitude: {band_rms_: .6f}")

***
##### Continuous Wavelet Transform (CWT)
- The Morlet wavelet, given by $\psi(t)=\exp(\frac{-t^2}{2})\cos(5t)$, is used for the wavelet transform
- The ssqueezepy library is used to compute the continuous wavelet transform

In [ ]:
# CWT - Drone WITH Payload:
with PdfPages('CWT_drone_with_payload.pdf') as pdf:
    signal1 = apply_hann_window(with_payload, 'detrended_centered') # use tapered, not normalized data for input
    time1 = with_payload['Time (s)']
    
    downsample_factor1 = 10 # downsample to speed up computation
    signal_ds1 = scipy.signal.decimate(signal1, downsample_factor1)
    sampling_rate_ds1 = sr1 // downsample_factor1 # note: sr = 48000 Hz
    time_ds1 = time1[::downsample_factor1]
    print(f"Sample rate after downsampling: {sampling_rate_ds1} Hz")
    print(f"Nyquist: {sampling_rate_ds1 / 2} Hz")
    
    # Setup Resolution:
    nv = 32 # number of voices/wavelets per octave
    # Perform CWT:
    Wx1, scales1 = sq.cwt(signal_ds1, 'morlet', nv=nv, fs=sampling_rate_ds1)
    frequencies1 = scale_to_freq(scales1, 'morlet', len(signal_ds1), fs=sampling_rate_ds1) # need to convert scales to frequencies for ssqueezepy
    
    # Max frequency should not exceed Nyquist --> limit freq range based on this:
    freq_mask1 = (frequencies1 >=5) & (frequencies1 <= sampling_rate_ds1 / 2)
    freqs_plot1 = frequencies1[freq_mask1]
    Wx_masked1 = Wx1[freq_mask1, :]         
    
    # Time downsample for display
    time_skip1 = 50
    Wx_plot1 = np.log10(np.abs(Wx_masked1[:, ::time_skip1])+ 1e-10) # add small epsilon to prevent log error
    time_display1 = time_ds1[::time_skip1]
    time_display_minutes1 = time_display1/60
    
    # Option: Define the "floor" for the data
    # e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
    threshold_val = -3.0 # note: vmin should be close to threshold value
    Wx_plot1[Wx_plot1 < threshold_val] = np.nan
    
    # Plot:
    plt.figure(figsize=(14, 6))
    mesh1 = plt.pcolormesh(time_display_minutes1, freqs_plot1, Wx_plot1, cmap='jet', shading='auto', vmax=-1.5, vmin=threshold_val)
    plt.yscale('log')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (min)')
    plt.title('Drone Audio With Payload')
    # Add colorbar
    plt.colorbar(mesh1, label='log\u2081\u2080(Magnitude)')
    pdf.savefig()
    plt.show()
    plt.close()

In [ ]:
with PdfPages('CWT_drone_without_payload.pdf') as pdf:
    # CWT - Drone WITHOUT Payload:
    signal2 = apply_hann_window(without_payload, 'detrended_centered')
    time2 = without_payload['Time (s)']
    
    downsample_factor2 = 10
    signal_ds2 = scipy.signal.decimate(signal2, downsample_factor2)
    sampling_rate_ds2 = sr2 // downsample_factor2 # note: sr = 48000 Hz
    time_ds2 = time2[::downsample_factor2]
    print(f"Sample rate after downsampling: {sampling_rate_ds2} Hz")
    print(f"Nyquist: {sampling_rate_ds2 / 2} Hz")
    
    # Setup Resolution:
    nv = 32
    # Perform CWT:
    Wx2, scales2 = sq.cwt(signal_ds2, 'morlet', nv=nv, fs=sampling_rate_ds2)
    frequencies2 = scale_to_freq(scales2, 'morlet', len(signal_ds2), fs=sampling_rate_ds2) # need to convert scales to frequencies for ssqueezepy
    
    # Max frequency should not exceed Nyquist --> limit freq range based on this:
    freq_mask2 = (frequencies2 >=5) & (frequencies2 <= sampling_rate_ds2 / 2)
    freqs_plot2 = frequencies2[freq_mask2]
    Wx_masked2 = Wx2[freq_mask2, :]         
    
    # Time downsample for display
    time_skip2 = 100
    Wx_plot2 = np.log10(np.abs(Wx_masked2[:, ::time_skip2])+ 1e-10) # add small epsilon to prevent log error
    time_display2 = time_ds2[::time_skip2]
    time_display_minutes2 = time_display2/60
    
    # Option: Define the "floor" for the data
    # e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
    threshold_val = -3.0 # note: vmin should be close to threshold value
    Wx_plot2[Wx_plot2 < threshold_val] = np.nan
    
    # Plot:
    plt.figure(figsize=(14, 6))
    mesh2 = plt.pcolormesh(time_display_minutes2, freqs_plot2, Wx_plot2, cmap='jet', shading='auto', vmax=-1.5, vmin=threshold_val)
    plt.yscale('log')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (min)')
    plt.title('Drone Audio Without Payload')
    # Add colorbar:
    plt.colorbar(mesh2, label='log\u2081\u2080(Magnitude)')
    pdf.savefig()
    plt.show()
    plt.close()

In [ ]:
# Making above CWT code into function:
def plot_cwt(dataframe, signal_column, time_column, sr, downsample, time_skip, title, threshold, vmin, vmax):
    """Plot CWT scalogram with Morlet wavelet.
    dataframe: pandas dataframe containing signal and time data
    signal_column: name of signal column in dataframe, str
    time_column: name of time (sec) column in dataframe, str
    sr: sample rate of dataframe data in Hz, int
    downsample: step for downsampling the signal, int
    time_skip: step size to downsample time for plot display, int
    title: title of plot, str
    threshold: any CWT magnitudes lower than the given threshold value will be treated as NaN
    vmin: set vmin for CWT plot, recommended to be a similar value to threshold
    vmax: set vmax for CWT plot"""
    with PdfPages(f'{title}.pdf') as pdf:
        signal = apply_hann_window(dataframe, signal_column) # uses previous function to taper at 10% from data edges
        time = dataframe[time_column]
        
        signal_ds = scipy.signal.decimate(signal, downsample)
        sampling_rate_ds = sr // downsample 
        time_ds = time[::downsample]
        print(f"Sample rate after downsampling: {sampling_rate_ds} Hz")
        print(f"Nyquist: {sampling_rate_ds / 2} Hz")
        
        # Setup Resolution:
        nv = 32
        # Perform CWT:
        Wx, scales = sq.cwt(signal_ds, 'morlet', nv=nv, fs=sampling_rate_ds)
        frequencies = scale_to_freq(scales, 'morlet', len(signal_ds), fs=sampling_rate_ds) # need to convert scales to frequencies for ssqueezepy
        
        # Max frequency should not exceed Nyquist --> limit freq range based on this:
        freq_mask = (frequencies >=5) & (frequencies <= sampling_rate_ds / 2)
        freqs_plot = frequencies[freq_mask]
        Wx_masked = Wx[freq_mask, :]         
        
        # Time downsample for display
        Wx_plot = np.log10(np.abs(Wx_masked[:, ::time_skip])+ 1e-10) # add small epsilon to prevent log error
        time_display = time_ds[::time_skip]
        time_display_minutes = time_display/60 # converts time to minutes, assuming time_column data is in seconds
        
        # e.g. anything quieter than 'threshold' will be treated as "Not a Number" (NaN), which renders as transparent/white.
        Wx_plot[Wx_plot < threshold] = np.nan
        
        # Plot:
        plt.figure(figsize=(14, 6))
        mesh = plt.pcolormesh(time_display_minutes, freqs_plot, Wx_plot, cmap='jet', shading='auto', vmax=vmax, vmin=vmin)
        plt.yscale('log')
        plt.ylabel('Frequency (Hz)')
        plt.xlabel('Time (min)')
        plt.title(title)
        # Add colorbar:
        plt.colorbar(mesh, label='log\u2081\u2080(Magnitude)')
        pdf.savefig()
        plt.show()
        plt.close()
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# test:
plot_cwt(without_payload, 'detrended_centered', 'Time (s)', 48000, 10, 100, 'Drone Audio Without Payload', -3.0, -3.0, -1.5)
    

##### CWT Difference Plot:
- Plot a scalogram for the difference between CWT plots for drone audio with payload and drone audio without a payload

In [ ]:
# CWT - Drone WITH Payload:
signal1 = apply_hann_window(with_payload, 'detrended_centered')
time1 = with_payload['Time (s)']
    
# CWT - Drone WITHOUT Payload:
signal2 = apply_hann_window(without_payload, 'detrended_centered')
time2 = without_payload['Time (s)']
    
downsample_factor1 = 10
downsample_factor2 = 10
    
signal_ds1 = scipy.signal.decimate(signal1, downsample_factor1)
signal_ds2 = scipy.signal.decimate(signal2, downsample_factor2)
    
# Trim to same length after decimating to prevent off-by-one errors in CWT
min_ds_len = min(len(signal_ds1), len(signal_ds2))
signal_ds1 = signal_ds1[:min_ds_len]
signal_ds2 = signal_ds2[:min_ds_len]
    
sampling_rate_ds1 = sr1 // downsample_factor1
sampling_rate_ds2 = sr2 // downsample_factor2
    
time_ds1 = time1[::downsample_factor1][:min_ds_len]
time_ds2 = time2[::downsample_factor2][:min_ds_len]
   
print(f"Sample rate after downsampling: {sampling_rate_ds1} Hz")
print(f"Nyquist: {sampling_rate_ds1 / 2} Hz")
  
# Setup Resolution:
nv = 32
   
# Perform CWT:
Wx1, scales1 = sq.cwt(signal_ds1, 'morlet', nv=nv, fs=sampling_rate_ds1)
frequencies1 = scale_to_freq(scales1, 'morlet', len(signal_ds1), fs=sampling_rate_ds1) # need to convert scales to frequencies for ssqueezepy

Wx2, scales2 = sq.cwt(signal_ds2, 'morlet', nv=nv, fs=sampling_rate_ds2)
frequencies2 = scale_to_freq(scales2, 'morlet', len(signal_ds2), fs=sampling_rate_ds2)
    
# Limit freq range to 5 Hz - Nyquist:
freq_mask1 = (frequencies1 >= 5) & (frequencies1 <= sampling_rate_ds1 / 2)
freq_mask2 = (frequencies2 >= 5) & (frequencies2 <= sampling_rate_ds2 / 2)
    
freqs_plot1 = frequencies1[freq_mask1]
freqs_plot2 = frequencies2[freq_mask2]
   
Wx_masked1 = Wx1[freq_mask1, :]
Wx_masked2 = Wx2[freq_mask2, :]
    
# Time downsample for display:
time_skip1 = 50
time_skip2 = 50
    
Wx_plot1 = np.log10(np.abs(Wx_masked1[:, ::time_skip1]) + 1e-10)
Wx_plot2 = np.log10(np.abs(Wx_masked2[:, ::time_skip2]) + 1e-10)
  
time_display1 = time_ds1[::time_skip1]
time_display2 = time_ds2[::time_skip2]
    
time_display_minutes1 = time_display1 / 60
time_display_minutes2 = time_display2 / 60
    
            
# Define the "floor" for the data, e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
threshold_val = -3.0
Wx_plot1[Wx_plot1 < threshold_val] = np.nan
Wx_plot2[Wx_plot2 < threshold_val] = np.nan
    
# Plot:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
 
im1 = axes[0].pcolormesh(time_display_minutes1, freqs_plot1, Wx_plot1, cmap='jet', shading='auto', vmax=-1.5, vmin=threshold_val)
axes[0].set_title("1: Drone Audio With Payload")
axes[0].set_xlabel("Time (min)")
axes[0].set_ylabel("Frequency (Hz)")
axes[0].set_yscale("log")
plt.colorbar(im1, ax=axes[0], label=r"$\log_{10}$(Magnitude)")
    
im2 = axes[1].pcolormesh(time_display_minutes2, freqs_plot2, Wx_plot2, cmap='jet', shading='auto', vmax=-1.5, vmin=threshold_val)
axes[1].set_title("2: Drone Audio Without Payload")
axes[1].set_xlabel("Time (min)")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_yscale("log")
plt.colorbar(im2, ax=axes[1], label=r"$\log_{10}$(Magnitude)")
    
# Difference map (no trimming needed, arrays are already the same length):
difference = Wx_plot1 - Wx_plot2
#max_diff = np.percentile(np.abs(difference), 98)
im3 = axes[2].pcolormesh(time_display_minutes1, freqs_plot1, difference, cmap='RdBu', shading='auto', vmin=-1.0, vmax=1.0)
axes[2].set_title('Difference (1-2)')
axes[2].set_xlabel('Time (min)')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_yscale('log')
plt.colorbar(im3, ax=axes[2], label=r"$\log_{10}$(Magnitude)")
    
plt.tight_layout()
plt.savefig('CWT_difference_plot.jpg')
plt.show()
plt.close()

In [ ]:
# Alternative plot: CWT with joined drone data
joined_signal = raw_joined['Raw Amplitude'] # With payload to without payload
joined_time = raw_joined['Time (s)']
    
downsample_factor3 = 10
signal_ds3 = scipy.signal.decimate(joined_signal, downsample_factor3)
sampling_rate_ds3 = 48000 // downsample_factor3 # note: sr = 48000 Hz
time_ds3 = joined_time[::downsample_factor3]
print(f"Sample rate after downsampling: {sampling_rate_ds3} Hz")
print(f"Nyquist: {sampling_rate_ds3 / 2} Hz")
    
# Setup Resolution:
nv = 32
# Perform CWT:
Wx3, scales3 = sq.cwt(signal_ds3, 'morlet', nv=nv, fs=sampling_rate_ds3)
frequencies3 = scale_to_freq(scales3, 'morlet', len(signal_ds3), fs=sampling_rate_ds3) # need to convert scales to frequencies for ssqueezepy
    
# Max frequency should not exceed Nyquist --> limit freq range based on this:
freq_mask3 = (frequencies3 >=5) & (frequencies3 <= sampling_rate_ds3 / 2)
freqs_plot3 = frequencies3[freq_mask3]
Wx_masked3 = Wx3[freq_mask3, :]         
    
# Time downsample for display
time_skip3 = 100
Wx_plot3 = np.log10(np.abs(Wx_masked3[:, ::time_skip3])+ 1e-10) # add small epsilon to prevent log error
time_display3 = time_ds3[::time_skip3]
time_display_minutes3 = time_display3/60
    
# Option: Define the "floor" for the data
# e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
threshold_val = -3.5
Wx_plot3[Wx_plot3 < threshold_val] = np.nan
    
# Plot:
plt.figure(figsize=(14, 6))
mesh3 = plt.pcolormesh(time_display_minutes3, freqs_plot3, Wx_plot3, cmap='jet', shading='auto', vmax=-1.5, vmin=threshold_val)
plt.yscale('log')
plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (min)')
plt.title('Joined Drone Audio - With Payload to Without Payload')
# Add colorbar:
plt.colorbar(mesh3, label='log\u2081\u2080(Magnitude)')
plt.savefig('joined_CWT.jpg')
plt.show()
plt.close()

***
##### Save Plots & Markdown as PDF:
- Cell outputs and markdown comments are saved as an .html file, which can be converted to a .pdf using Ctrl+P

In [ ]:
# OLD VERSION
# Save notebook as .html file
# def save_notebook(notebook_name, output_name=None):
#     """Save Jupyter notebook outputs and markdown to .html file (excludes code cells)
#     notebook_name : str, name of the notebook file (e.g., 'notebook.ipynb')
#     output_name : str, name for the output file (optional). If None, uses the notebook name with .html extension"""
    
#     # Set output file name
#     if output_name is None:
#         output_name = notebook_name.replace('.ipynb', '.html')
    
#     # Check if notebook file exists
#     if not os.path.exists(notebook_name):
#         raise FileNotFoundError(f"Notebook file '{notebook_name}' not found")
            
#     result = subprocess.run(['jupyter', 'nbconvert', '--to', 'html', '--no-input',  # Exclude code cells
#     notebook_name, '--output', output_name.replace('.html', '')], capture_output=True, text=True)
            
#     if result.returncode == 0: # return code of zero indicates successful execution 
#         print(f"Created HTML file: {output_name}")
#         print("Open this in your browser and use Ctrl+P --> Save as PDF")
#         print("(The HTML file contains only outputs and markdown, no code cells)")
#     else:
#         print("Error:")
#         print(result.stderr) # standard error

# # Example usage:
# # save_notebook('notebook.ipynb')
# # Or specify custom output name:
# # save_notebook('notebook.ipynb', 'output_notebook.html')

##### Exporting Computed Values:
- Goal: Export all computed values to an organized folder
- Note: Make sure this is done in a cell at the END of the notebook

In [ ]:
import os
from datetime import datetime # datetime is used so each export gets its own folder (i.e. old exports are not overwritten)

# Helper functions:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ 
def _mkdir(path):
    """Creates a directory at "path," including any missing parent folders
    Returns the path string so it can be used inline (e.g. d = _mkdir("exports/subfolder")."""
    os.makedirs(path, exist_ok=True)
    # Notes:
    # Creates a directory if it does not already exist. 
    # The exist_ok=True argument prevents an error if the folder is already there.
    return path

def save_txt(array, path, header="", fmt="%.8e"):
    """Save 1-D or 2-D real array as a .txt file.
    array: real-valued numpy array containing the data to save
    path: full file path (string), should end in ".txt"
    header: data description for top of file (string)
    fmt: number format for each value (string)"""
    np.savetxt(path, array, header=header, fmt=fmt, comments="# ")
    # Notes: 
    # Writes a numpy array to a plain-text file, one row per line. Columns are separated by spaces (or a delimiter you choose).
    # The header parameter writes comment lines at the top of the file so you know what each column is.
    # The fmt parameter controls how many digits are written. "%.8e" means 8 decimal places in scientific notation.
    # Can reload later, e.g. data = np.loadtxt("file.txt")

def save_npy(array, path):
    """Save any array (including complex-valued) as a binary .npy file.
    array: numpy array, can be real, complex, any dtype or shape
    path: full file path (string), should end in '.npy' """
    np.save(path, array)
    # Notes:
    # Writes a numpy array to a binary .npy file
    # Preserves complex numbers (unlike savetxt()), arbitrary dtypes, etc.
    # Better file type for Wx coefficients from CWT
    # e.g. to reload later:
    # Wx = np.load("Wx_with_payload.npy")
    # magnitude = np.abs(Wx)
    # phase = np.angle(Wx)

# output directory:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") # format the date/time
OUT = _mkdir(f"exports/drone_analysis_{timestamp}")

print(f"Exporting to: {OUT}/")
print("=" * 55)

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 1. RAW DATA
d = _mkdir(f"{OUT}/01_raw_data")

save_txt(np.column_stack([raw_with["Time (s)"], raw_with["Raw Amplitude"]]), f"{d}/raw_with_payload.txt",header="time_s  raw_amplitude")

save_txt(np.column_stack([raw_without["Time (s)"], raw_without["Raw Amplitude"]]), f"{d}/raw_without_payload.txt", header="time_s  raw_amplitude")

save_txt(np.column_stack([raw_joined["Time (s)"], raw_joined["Raw Amplitude"]]), f"{d}/raw_joined.txt",
         header="time_s  raw_amplitude  (with-payload first, then without-payload)")
print("Done: 01_raw_data")

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 2. DNS DATA  (Detrended, Normalized, Mean-Shifted, after align & trim)
d = _mkdir(f"{OUT}/02_dns_data")

save_txt(np.column_stack([with_payload["Time (s)"],
        with_payload["Amplitude"],          # normalised DNS
        with_payload["detrended_centered"]  # non-normalized, detrended+centred
    ]), f"{d}/dns_with_payload.txt", header="time_s  amplitude_normalised  amplitude_detrended_centred")

save_txt(np.column_stack([without_payload["Time (s)"], without_payload["Amplitude"], without_payload["detrended_centered"]]),
    f"{d}/dns_without_payload.txt", header="time_s  amplitude_normalised  amplitude_detrended_centred")
print("Done: 02_dns_data")

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 3. TAPERED DATA  (Hann-edge taper, 10% from each side)
d = _mkdir(f"{OUT}/03_tapered_data")

save_txt(np.column_stack([with_payload["Time (s)"], with_p_tapered]), f"{d}/tapered_with_payload.txt",
    header="time_s  tapered_amplitude  (Hann window, taper_fraction=0.10)")

save_txt(np.column_stack([without_payload["Time (s)"], without_p_tapered]), f"{d}/tapered_without_payload.txt",
    header="time_s  tapered_amplitude  (Hann window, taper_fraction=0.10)")
print("Done: 03_tapered_data")

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 4. AVERAGED FFT
# Four sets: DNS (seg=8096) and non-normalized (seg=4096) x with/without
d= _mkdir(f"{OUT}/04_averaged_fft")

# DNS, segment_size = 8096
save_txt(np.column_stack([freqs_1, avg_1]), f"{d}/avg_fft_dns_with_payload_seg8096.txt",
         header="frequency_hz  avg_magnitude  (DNS, seg=8096, 50%-overlap Hann, averaged)")

save_txt(np.column_stack([freqs_2, avg_2]),f"{d}/avg_fft_dns_without_payload_seg8096.txt",
    header="frequency_hz  avg_magnitude  (DNS, seg=8096, 50%-overlap Hann, averaged)")

# Non-normalized (detrended_centred), segment_size = 4096
save_txt(np.column_stack([freqs__1, avg__1]), f"{d}/avg_fft_nonnorm_with_payload_seg4096.txt",
    header="frequency_hz  avg_magnitude  (detrended_centred, seg=4096, 50%-overlap Hann, averaged)")

save_txt(np.column_stack([freqs__2, avg__2]),f"{d}/avg_fft_nonnorm_without_payload_seg4096.txt",
    header="frequency_hz  avg_magnitude  (detrended_centred, seg=4096, 50%-overlap Hann, averaged)")
print("Done: 04_averaged_fft")


#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 5. WELCH PSD
d = _mkdir(f"{OUT}/05_welch_psd")

save_txt(np.column_stack([freqs_with, psd_with]), f"{d}/psd_with_payload.txt",
         header="frequency_hz  psd  (Welch, seg=4096, 50%-overlap Hann, scaling=density)")

save_txt(np.column_stack([freqs_without, psd_without]), f"{d}/psd_without_payload.txt",
    header="frequency_hz  psd  (Welch, seg=4096, 50%-overlap Hann, scaling=density)")
print("Done: 05_welch_psd")

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 6. CWT - Individual scalograms
# Exports:
# -the full complex coefficient matrix  Wx  (binary .npy)
# -log10 magnitude display matrix  Wx_plot  (.txt file)
# -frequency axis (.txt)
# -display time axis (.txt, in minutes)
d = _mkdir(f"{OUT}/06_cwt_individual")

# WITH payload:
save_npy(Wx1, f"{d}/Wx_with_payload_complex.npy")                  # shape: (n_freq, n_time_ds)
save_npy(Wx_masked1, f"{d}/Wx_masked_with_payload_complex.npy")    # shape: (n_freq_masked, n_time_ds)
save_txt(Wx_plot1, f"{d}/Wx_plot_with_payload_log10mag.txt",
         header="log10(|CWT|) display matrix - rows=frequency, cols=time  (NaN below threshold=-3.0)")
save_txt(freqs_plot1.reshape(-1, 1), f"{d}/frequencies_with_payload_hz.txt", header="frequency_hz  (5 Hz to Nyquist, frequency-masked)")
save_txt(np.array(time_display_minutes1).reshape(-1, 1), f"{d}/time_display_with_payload_min.txt",
         header="time_min  (display axis, after time_skip downsampling)")

# WITHOUT payload:
save_npy(Wx2, f"{d}/Wx_without_payload_complex.npy")
save_npy(Wx_masked2, f"{d}/Wx_masked_without_payload_complex.npy")
save_txt(Wx_plot2, f"{d}/Wx_plot_without_payload_log10mag.txt",
         header="log10(|CWT|) display matrix - rows=frequency, cols=time  (NaN below threshold=-3.0)")
save_txt(freqs_plot2.reshape(-1, 1), f"{d}/frequencies_without_payload_hz.txt",
         header="frequency_hz  (5 Hz to Nyquist, frequency-masked)")
save_txt(np.array(time_display_minutes2).reshape(-1, 1), f"{d}/time_display_without_payload_min.txt",
         header="time_min  (display axis, after time_skip downsampling)")

print("Done: 06_cwt_individual")

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ 
# 7. CWT DIFFERENCE PLOT 
d = _mkdir(f"{OUT}/07_cwt_difference")

save_txt(difference, f"{d}/cwt_difference_log10mag.txt", header="CWT difference matrix: log10|Wx_with| - log10|Wx_without|, rows=frequency, cols=time")
save_txt(freqs_plot1.reshape(-1, 1), f"{d}/frequencies_hz.txt", header="frequency_hz  (shared frequency axis for both scalograms)")
save_txt(np.array(time_display_minutes1).reshape(-1, 1), f"{d}/time_display_min.txt", header="time_min  (shared display time axis)")
print("Done: 07_cwt_difference")

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 8. CWT, JOINED signal
d = _mkdir(f"{OUT}/08_cwt_joined")

save_npy(Wx3, f"{d}/Wx_joined_complex.npy")
save_npy(Wx_masked3, f"{d}/Wx_masked_joined_complex.npy")
save_txt(Wx_plot3, f"{d}/Wx_plot_joined_log10mag.txt", header="log10(|CWT|) joined signal, rows=frequency, cols=time  (NaN below threshold=-3.5)")
save_txt(freqs_plot3.reshape(-1, 1), f"{d}/frequencies_joined_hz.txt", header="frequency_hz")
save_txt(np.array(time_display_minutes3).reshape(-1, 1), f"{d}/time_display_joined_min.txt", header="time_min")
print("Done: 08_cwt_joined")

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Summary:
# describes every export for reference
with open(f"{OUT}/summary.txt", "w") as f:
    f.write(f"Drone With/Without Payload - Export Summary\n")
    f.write(f"{'='*55}\n")
    f.write(f"Timestamp         : {timestamp}\n")
    f.write(f"Sampling rate     : {sr1} Hz\n")
    f.write(f"DNS samples (each): {len(with_payload)}\n")
    f.write(f"Taper fraction    : 10% (Hann window)\n")
    f.write(f"CWT wavelet       : Morlet, nv=32 voices/octave\n")
    f.write(f"CWT downsample    : 10x -> {sr1 // 10} Hz\n\n")
    f.write("Folders\n")
    f.write("-------\n")
    f.write("01_raw_data/           Raw amplitude vs time (with, without, joined)\n")
    f.write("02_dns_data/           Detrended, normalised, mean-shifted + aligned/trimmed\n")
    f.write("03_tapered_data/       Hann-edge-tapered amplitude\n")
    f.write("04_averaged_fft/       Averaged FFT magnitude (DNS seg=8096, non-norm seg=4096)\n")
    f.write("05_welch_psd/          Welch PSD (seg=4096, 50%-overlap Hann)\n")
    f.write("06_cwt_individual/     CWT scalograms - with & without payload\n")
    f.write("07_cwt_difference/     CWT difference map (with - without)\n")
    f.write("08_cwt_joined/         CWT of joined (concatenated) signal\n\n")
    f.write("Notes\n")
    f.write("-----\n")
    f.write("*.txt files  - load with np.loadtxt()\n")
    f.write("*.npy files  - complex CWT matrices, load with np.load()\n")
    f.write("2-D .txt matrices: rows = frequency bins, cols = time bins\n")

print(f"\nDone: summary.txt")
print(f"\nAll exports saved to: {OUT}/")